<a href="https://colab.research.google.com/github/OutisAyo/council-classifier-/blob/main/notebooks/16_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning for Automated Classification and Prioritisation of Local Council Service Requests in the UK
## Notebook 16 Streamlit Dashboard

**Author:** Fashina Fuad Ayomide  
**MSc Data Science, University of South Wales**

This notebook creates and launches the Streamlit dashboard that integrates the department classifier, priority predictor, and decision support layer into a single interactive interface.

## Mounting Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Installing Streamlit and localtunnel

In [33]:
!pip install streamlit -q
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧

## Writing the dashboard script

The entire dashboard is written to a single app.py file. Streamlit reads this file and turns it into a web interface.

In [34]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import pickle
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime

# ============================================================
# Page configuration
# ============================================================
st.set_page_config(
    page_title="Council Request Classifier",
    page_icon="🏛️",
    layout="wide"
)

# ============================================================
# Loading models and components (cached so they load only once)
# ============================================================
MODELS_DIR = '/content/drive/MyDrive/council-classifier/models'

@st.cache_resource
def load_components():
    department_pipeline = joblib.load(f'{MODELS_DIR}/department_pipeline_rf.pkl')
    priority_pipeline = joblib.load(f'{MODELS_DIR}/priority_pipeline_final.pkl')
    retrieval_vectorizer = joblib.load(f'{MODELS_DIR}/retrieval_vectorizer.pkl')
    with open(f'{MODELS_DIR}/response_templates.pkl', 'rb') as f:
        templates = pickle.load(f)
    with open(f'{MODELS_DIR}/urgency_lexicon.pkl', 'rb') as f:
        urgency_lexicon = pickle.load(f)
    dataset = pd.read_pickle(f'{MODELS_DIR}/dataset_for_retrieval.pkl')
    historical_vectors = retrieval_vectorizer.transform(dataset['request_text'])
    return department_pipeline, priority_pipeline, retrieval_vectorizer, templates, urgency_lexicon, dataset, historical_vectors

department_pipeline, priority_pipeline, retrieval_vectorizer, templates, urgency_lexicon, dataset, historical_vectors = load_components()

ALL_URGENCY_WORDS = [w for group in urgency_lexicon.values() for w in group]

def detect_urgency(text):
    text_lower = text.lower()
    return [w for w in ALL_URGENCY_WORDS if w in text_lower]

# ============================================================
# Decision support functions
# ============================================================
def find_similar_requests(new_text, top_n=5):
    new_vector = retrieval_vectorizer.transform([new_text])
    similarities = cosine_similarity(new_vector, historical_vectors).flatten()
    top_indices = similarities.argsort()[-top_n:][::-1]
    similar = dataset.iloc[top_indices].copy()
    similar['similarity_score'] = similarities[top_indices]
    valid = similar[
        (similar['date_closed'].notna()) &
        ((similar['date_closed'] - similar['date_received']).dt.days >= 0)
    ]
    avg_days = (valid['date_closed'] - valid['date_received']).dt.days.mean() if len(valid) > 0 else None
    return similar[['request_text', 'assigned_department', 'priority', 'similarity_score']], avg_days

def check_for_duplicates(new_text, recent_days=7, similarity_threshold=0.7, top_n=5):
    most_recent = dataset['date_received'].max()
    cutoff = most_recent - pd.Timedelta(days=recent_days)
    recent = dataset[dataset['date_received'] >= cutoff].copy()
    if len(recent) == 0:
        return pd.DataFrame(), False
    recent_vectors = retrieval_vectorizer.transform(recent['request_text'])
    new_vector = retrieval_vectorizer.transform([new_text])
    similarities = cosine_similarity(new_vector, recent_vectors).flatten()
    recent['similarity_score'] = similarities
    duplicates = recent[recent['similarity_score'] >= similarity_threshold]
    duplicates = duplicates.sort_values('similarity_score', ascending=False).head(top_n)
    return duplicates[['request_text', 'assigned_department', 'date_received', 'similarity_score']], len(duplicates) > 0

# ============================================================
# Dashboard layout
# ============================================================
st.title("🏛️ UK Council Service Request Classifier")
st.markdown("**Automated classification, prioritisation and decision support for local council service requests**")
st.markdown("---")

tab1, tab2 = st.tabs(["📝 Classify a Request", "📊 Data Insights"])

# ============================================================
# TAB 1 — Classification interface
# ============================================================
with tab1:
    col1, col2 = st.columns([2, 1])

    with col1:
        request_text = st.text_area(
            "Enter the service request text:",
            placeholder="e.g. live rat in house urgent",
            height=100
        )

    with col2:
        submission_date = st.date_input("Submission date:", datetime.today())

    if st.button("Classify Request", type="primary"):
        if request_text.strip():
            day_of_week = submission_date.weekday()
            month = submission_date.month

            predicted_department = department_pipeline.predict([request_text])[0]
            priority_input = pd.DataFrame({
                'request_text': [request_text],
                'day_of_week': [day_of_week],
                'month': [month]
            })
            model_priority = priority_pipeline.predict(priority_input)[0]
            urgency_matches = detect_urgency(request_text)

            if urgency_matches:
                final_priority = 'HIGH'
            else:
                final_priority = model_priority

            m1, m2 = st.columns(2)
            m1.metric("Predicted Department", predicted_department)
            m2.metric("Predicted Priority", final_priority)

            if urgency_matches:
                st.warning(f"⚠️ Escalated to HIGH — urgency language detected: {', '.join(urgency_matches)}")

            template = templates.get(predicted_department, {}).get(
                final_priority,
                "Thank you for your request. It has been logged and will be reviewed."
            )

            similar, avg_resolution = find_similar_requests(request_text)
            context = f" Based on similar requests, this typically takes around {avg_resolution:.0f} days to resolve." if avg_resolution else ""

            st.subheader("Suggested Response")
            st.info(template + context)

            duplicates, is_duplicate = check_for_duplicates(request_text)
            st.subheader("Duplicate Check")
            if is_duplicate:
                st.warning(f"⚠️ {len(duplicates)} similar request(s) submitted recently — possible duplicate.")
                st.dataframe(duplicates, use_container_width=True)
            else:
                st.success("✅ No similar recent requests found.")

            st.subheader("Most Similar Historical Requests")
            st.dataframe(similar, use_container_width=True)
        else:
            st.error("Please enter a request text first.")

# ============================================================
# TAB 2 — Data insights
# ============================================================
with tab2:
    st.subheader("Dataset Overview")
    c1, c2, c3 = st.columns(3)
    c1.metric("Total Records", f"{len(dataset):,}")
    c2.metric("Departments", dataset['assigned_department'].nunique())
    c3.metric("Date Range", f"{dataset['date_received'].min().year} - {dataset['date_received'].max().year}")

    st.subheader("Requests by Department")
    dept_counts = dataset['assigned_department'].value_counts()
    st.bar_chart(dept_counts)

    st.subheader("Priority Distribution")
    prio_counts = dataset['priority'].value_counts()
    st.bar_chart(prio_counts)

    st.subheader("Monthly Request Volume")
    monthly = dataset.groupby(dataset['date_received'].dt.to_period('M')).size()
    monthly.index = monthly.index.astype(str)
    st.line_chart(monthly)

Overwriting app.py


## Launching the dashboard

We run Streamlit in the background and open a public tunnel to view it.

In [46]:
!streamlit run app.py &>/content/logs.txt &
!npx localtunnel --port 8501 & curl https://loca.lt/mytunnelpassword

⠙⠹⠸⠼⠴⠦⠧34.75.126.43your url is: https://old-yaks-fly.loca.lt


In [64]:
!pkill -f cloudflared
!pkill -f streamlit
import time
time.sleep(5)
print("Cleaned up")

Cleaned up


In [65]:
!streamlit run app.py &>/content/logs.txt &
print("Streamlit starting...")

Streamlit starting...


In [66]:
import time
for i in range(6):
    time.sleep(10)
    result = !curl -s -o /dev/null -w "%{http_code}" http://localhost:8501
    print(f"Attempt {i+1}: HTTP {result[0]}")
    if result[0] == "200":
        print("✅ Streamlit is ready")
        break

Attempt 1: HTTP 200
✅ Streamlit is ready


In [ ]:
!./cloudflared tunnel --url http://localhost:8501 --no-autoupdate 2>&1 | grep -o 'https://.*trycloudflare.com'

https://challenge-badly-lectures-incredible.trycloudflare.com
